# Ticker-to-Thesis: run the engine on real data

This notebook pulls **10-K/10-Q data from SEC EDGAR** for a company and its peers, benchmarks it, values it
(DCF + comps + scenarios) and writes a **live-formula Excel model** and a **pitch deck**.

1. Set `SEC_USER_AGENT` below to your name and email (the SEC requires this; no API key needed).
2. **Runtime → Run all**. It takes about a minute.
3. The last cell downloads the Excel model and the deck.

Set `DEMO = True` to run offline on fictional companies instead.

In [1]:
REPO = "https://github.com/Abhishek10-r/ticker-to-thesis"
SEC_USER_AGENT = "Abhishek Raut raut.abhishek1010@gmail.com"                  # <- required by the SEC: your name and email
RUN_FILE = "config/runs/nike.yaml"                                   # ticker, peers, analyst, your own view
DEMO = False

import os, sys, subprocess
if not os.path.exists("t2t") and not os.path.exists("ticker-to-thesis"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if os.path.exists("ticker-to-thesis"): os.chdir("ticker-to-thesis")
subprocess.run(["git", "pull", "-q"])          # pick up the latest version of the code
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
print("ready in", os.getcwd())

ready in /content/ticker-to-thesis


## 1. Pull filings and prices, run the analysis

In [2]:
import yaml, pandas as pd
from t2t.pipeline import load_config, run, write_outputs
from t2t.sec import SECClient, FixtureSource
from t2t.market import FixtureMarketData
from t2t.quality import quality_table
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

spec = yaml.safe_load(open(RUN_FILE))
cfg = load_config(extra=spec.get("config_overrides"))
if DEMO:
    src, mkt, note = FixtureSource("tests/fixtures"), FixtureMarketData("tests/fixtures"), "DEMO DATA: fictional companies, for testing only."
    spec.update(ticker="DEMO", peers=["PEERA", "PEERB", "PEERC", "PEERD", "PEERE"])
else:
    src, mkt, note = SECClient(SEC_USER_AGENT), None, ""
rep = run(spec["ticker"], spec["peers"], src, mkt, cfg, overrides=spec.get("overrides"), analyst=spec.get("analyst", ""),
          analyst_view={k: v for k, v in (spec.get("analyst_view") or {}).items() if v}, data_note=note)
v = rep.val
print(f"\n{rep.name} ({rep.target}): {v.rating} | price target ${v.price_target:,.2f} vs ${v.price:,.2f} ({v.upside:+.1%})")
for w in rep.warnings: print("note:", w)

[sec] NKE: fetching filings
[sec] LULU: fetching filings
[sec] DECK: fetching filings
[sec] CROX: fetching filings
[sec] UAA: fetching filings
[sec] VFC: fetching filings
[sec] COLM: fetching filings
[market] prices and risk-free rate
[analysis] benchmarking, DCF, comps, scenarios

NIKE, Inc. (NKE): HOLD | price target $32.81 vs $33.87 (-3.1%)
note: NKE: no OperatingIncomeLoss tag for FY [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]; EBIT derived as gross profit − SG&A − R&D.


## 2. Data-quality check (look at this before using the output)
One row per company, latest fiscal year. Anything flagged deserves a look in the **Sources** sheet of the Excel model,
which shows the exact XBRL tag and SEC filing behind every number.

In [3]:
quality_table(rep)

,FY,revenue $m,EBIT margin,net income $m,FCF $m,cash $m,debt $m,shares m,price $,flags
ticker,,,,,,,,,,
NKE,2026,"46,398.00",0.08,"3,108.00","2,184.00","6,903.00","7,893.00","1,484.20",33.87,see notes
LULU,2025,"11,102.60",0.20,"1,579.18",921.67,"1,389.74",0.00,105.59,94.46,ok
DECK,2026,"5,472.30",0.23,"1,024.07","1,097.33","1,602.59",0.00,136.41,79.12,ok
CROX,2025,"4,040.65",0.04,-81.20,659.20,170.28,"1,334.00",47.95,118.09,ok
UAA,2026,"4,966.37",-0.03,-495.64,-162.16,395.98,591.16,431.94,4.71,ok
VFC,2026,"9,605.21",0.06,254.92,556.57,823.94,"3,530.01",391.77,14.33,ok
COLM,2025,"3,397.35",0.06,177.22,216.74,532.30,0.00,51.19,57.27,ok


## 3. Benchmarking and valuation

In [4]:
sc = rep.bench.scorecard[["label", "target", "peer_median", "percentile", "verdict"]]
display(sc)
for k, lines in rep.insights().items():
    if k in ("financial", "industry", "valuation", "risks"):
        print(f"\n{k.upper()}"); [print(" -", x) for x in lines]
display(v.football)
display(v.sens_growth.round(2))

,label,target,peer_median,percentile,verdict
metric,,,,,
revenue_growth,Revenue growth,0.00,0.01,35.71,Worse
revenue_cagr_3y,Revenue CAGR (3y),-0.03,0.02,35.71,Worse
gross_margin,Gross margin,0.43,0.56,7.14,Worse
ebit_margin,EBIT margin,0.08,0.06,64.29,Better
roic,ROIC,0.23,0.13,64.29,Better
fcf_margin,FCF margin,0.05,0.07,21.43,Worse
fcf_conversion,FCF conversion (FCF / net income),0.70,1.15,30.00,Worse
net_debt_ebitda,Net debt / EBITDA,-0.24,-0.67,41.67,Worse
interest_coverage,Interest coverage (EBIT / interest),11.76,3.46,58.33,Better



FINANCIAL
 - Growth: revenue grew 0.2% in the latest year vs a peer median of +1.0%; the 3-year CAGR is -3.2% vs +1.9% for peers.
 - Profitability: EBIT margin of 8.2% ranks at the 64th percentile of the group (peer median 6.0%). Over three years it moved -3.4pp while the peer median moved -7.5pp.
 - Returns: ROIC of 23.4% vs a peer median of 13.0% (above the group).
 - Cash: FCF margin 4.7% (peers 7.3%); FCF conversion 70% of net income.
 - Balance sheet: net cash (-0.2x) vs a peer median of net cash (-0.7x).

INDUSTRY
 - Peer-group revenue growth has averaged +0.3% a year over the last three years (latest +1.1%); NKE averaged -3.1%.
 - Industry EBIT margins are contracting (-7.5pp over three years, now 6.0% at the median).
 - Gross margin trend: peers +3.5pp, NKE -0.6pp over three years.

RISKS
 - Margin compression: EBIT margin down 3.4pp over three years; the forecast assumes a recovery, which is the key swing factor.
 - Valuation leans on the terminal value (77% of DCF enterprise

,method,low,high
0,52-week trading range (closing prices),33.87,71.93
1,Trading comps: EV/EBITDA 6.9x–9.7x,21.34,30.04
2,Trading comps: P/E 9.5x–17.9x,19.86,37.51
3,"DCF: perpetuity growth (WACC ±0.5pp, g ±0.25pp)",32.13,39.88
4,"DCF: exit multiple (WACC ±0.5pp, multiple ±1x)",29.59,36.54
5,Scenarios: bear to bull (DCF),22.50,58.53


,0.02,0.02,0.03,0.03,0.03
wacc,,,,,
0.08,39.19,40.55,42.04,43.66,45.43
0.09,36.16,37.31,38.54,39.88,41.33
0.09,33.56,34.53,35.57,36.69,37.90
0.10,31.30,32.13,33.02,33.97,34.99
0.10,29.32,30.04,30.80,31.61,32.49


## 4. Write and download the Excel model and pitch deck

In [5]:
files = write_outputs(rep, "output")
print(files)
try:
    from google.colab import files as colab_files
    for p in files.values(): colab_files.download(str(p))
except ImportError:
    pass

{'excel': PosixPath('output/NKE_2026-10-05_valuation_model.xlsx'), 'deck': PosixPath('output/NKE_2026-10-05_pitch_deck.pptx')}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>